# Task 2: Custom Byte-Pair Encoding (BPE) Tokenizer & Autoregressive Causal LM


## Objective

Build a simple custom BPE vocabulary and train a small causal language model with a custom causal mask.


## Short Theory

BPE repeatedly merges the most frequent adjacent token pair. A causal LM predicts the next token without seeing future positions.


## Step 1: Import Libraries


In [1]:
import re
from collections import Counter
import torch
import torch.nn as nn


## Step 2: Custom BPE Merge


In [3]:
text = "low lower lowest low low lower"
tokens = [list(w) + ["</w>"] for w in text.split()]
for _ in range(5):
    pairs = Counter((w[i], w[i+1]) for w in tokens for i in range(len(w)-1))
    if not pairs: break
    a, b = pairs.most_common(1)[0][0]
    for w in tokens:
        i = 0
        while i < len(w)-1:
            if w[i] == a and w[i+1] == b:
                w[i:i+2] = [a+b]
            else:
                i += 1
print("BPE tokens:", tokens)


BPE tokens: [['low</w>'], ['lower', '</w>'], ['lowe', 's', 't', '</w>'], ['low</w>'], ['low</w>'], ['lower', '</w>']]


## Step 3: Causal LM


In [4]:
vocab = {t:i for i,t in enumerate(sorted({t for w in tokens for t in w}))}
V = len(vocab); d = 32; T = 8
model = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(d_model=d, nhead=4, batch_first=True),
    num_layers=1
)
embed = nn.Embedding(V, d)
head = nn.Linear(d, V)
x = torch.randint(0, V, (4, T))
mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
h = model(embed(x), mask=mask)
logits = head(h)
print("Logits:", logits.shape)


Logits: torch.Size([4, 8, 6])


## Small Experiment

Generation Mask Check


In [5]:
print("Causal mask shape:", mask.shape)
print(mask.int())


Causal mask shape: torch.Size([8, 8])
tensor([[0, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.int32)


## Conclusion

Implemented a custom BPE merge process and a causal language-model forward pass with look-ahead masking.
